# Project 2 - Utilization of hospital beds during epidemics

## Primary Task

In [146]:
import numpy as np
import heapq
from tqdm import tqdm
from scipy.optimize import brentq
import itertools
import pandas as pd

In [253]:
# Set parameters - bed configuration
total_beds = 75

bedsA = 20
bedsB = 20
bedsC = 35

# Total time
T = 365

rng = np.random.default_rng()

In [255]:
# Arrival rate functions

# Ward A - Arrival rate for regular epidemic patients
def rateA(t):

    if t < 0 or t > 365:
        raise ValueError("t must be between 0 and 365")

    # Avoid division by zero at t = 0
    if t == 0:
        return -(1/3650)*t**2 + (1/10)*t + 1

    return -(1/3650)*t**2 + (1/10)*t


# Ward B - Arrival rate for intensive care patients
def rateB(t):
   
    if t < 0 or t > 365:
        raise ValueError("t must be between 0 and 365")

    return (1/5) * rateA(t)


# Ward C - Constant arrival rate
def rateC(t):
    
    return 6

In [256]:
# Length of stay (LOS) parameters
muA = np.log(4*np.sqrt(2))
muB = np.log(6*np.sqrt(2))
muC = np.log(5*np.sqrt(2))

sigmaA = np.sqrt(np.log(2))
sigmaB = np.sqrt(np.log(2))
sigmaC = np.sqrt(np.log(2))

In [ ]:
# Simulation function

def simulate_hospital(bedsA, bedsB, bedsC, T):

    # Generate arrivals
    arrivals_A = []

    t = 0.0

    while t < T:

        # Generate next interarrival time
        t += rng.exponential(scale=1.0 / rateA(t))

        if t < T:
            arrivals_A.append(t)

    arrivals_B = []

    t = 0.0

    while t < T:

        t += rng.exponential(scale=1.0 / rateB(t))

        if t < T:
            arrivals_B.append(t)

    arrivals_C = []

    t = 0.0

    while t < T:

        t += rng.exponential(scale=1.0 / rateC(t))

        if t < T:
            arrivals_C.append(t)

    # Make chronological event list
    events = []

    for t in arrivals_A:
        events.append((t, "A"))

    for t in arrivals_B:
        events.append((t, "B"))

    for t in arrivals_C:
        events.append((t, "C"))

    # Sort all arrivals by time
    events.sort(key=lambda x: x[0])

    # Arrival count
    num_arrivals_A = len(arrivals_A)
    num_arrivals_B = len(arrivals_B)
    num_arrivals_C = len(arrivals_C)

    # Departure heaps
    # Each heap contains discharge times for patients currently occupying beds
    deps_A = []
    deps_B = []
    deps_C = []

    # Performance measures
    blocked_A = 0
    blocked_B = 0
    blocked_C = 0

    # Ward-full counters (NEW: for correct P_block definition)
    ward_full_A = 0
    ward_full_B = 0
    ward_full_C = 0

    transferred_B_to_A = 0

    # Utilization variables
    area_A = 0.0
    area_B = 0.0
    area_C = 0.0

    last_time = 0.0

    # Main simulation loop
    for t, ward in events:

        # Remove discharged patients before handling the new arrival
        while deps_A and deps_A[0] <= t:
            heapq.heappop(deps_A)

        while deps_B and deps_B[0] <= t:
            heapq.heappop(deps_B)

        while deps_C and deps_C[0] <= t:
            heapq.heappop(deps_C)

        # Calculate utilization statistics
        occ_A = len(deps_A)
        occ_B = len(deps_B)
        occ_C = len(deps_C)

        dt = t - last_time

        area_A += occ_A * dt
        area_B += occ_B * dt
        area_C += occ_C * dt

        last_time = t

        # Ward A arrival
        if ward == "A":

            # Ward full indicator (NEW)
            if len(deps_A) >= bedsA:
                ward_full_A += 1

            if len(deps_A) < bedsA:

                # Admit patient
                los = rng.lognormal(muA, sigmaA)

                discharge_time = t + los

                heapq.heappush(deps_A, discharge_time)

            else:

                # No available bed
                blocked_A += 1

        # Ward B arrival
        elif ward == "B":

            # Ward full indicator (NEW)
            if len(deps_B) >= bedsB:
                ward_full_B += 1

            if len(deps_B) < bedsB:

                # Admit to ICU ward
                los = rng.lognormal(muB, sigmaB)

                discharge_time = t + los

                heapq.heappush(deps_B, discharge_time)

            elif len(deps_A) < bedsA:

                # ICU ward full -> overflow to Ward A
                los = rng.lognormal(muB, sigmaB)

                discharge_time = t + los

                heapq.heappush(deps_A, discharge_time)

                transferred_B_to_A += 1

            else:

                # Both wards full
                blocked_B += 1

        # Ward C arrival
        elif ward == "C":

            # Ward full indicator (NEW)
            if len(deps_C) >= bedsC:
                ward_full_C += 1

            if len(deps_C) < bedsC:

                los = rng.lognormal(muC, sigmaC)

                discharge_time = t + los

                heapq.heappush(deps_C, discharge_time)

            else:

                blocked_C += 1
    
    
    # Compute utilizations
    util_A = area_A / (bedsA * T)
    util_B = area_B / (bedsB * T)
    util_C = area_C / (bedsC * T)
    
    
    # Return performance measures
    return {
    
    # Patient who were blocked entirely - could not be admitted to the hospital
    "blocked_A": blocked_A,
    "blocked_B": blocked_B,
    "blocked_C": blocked_C,

    # Total blocked
    "Total_blocked": blocked_A + blocked_B + blocked_C,

    # Number of transfers from B to A
    "transferred_B_to_A": transferred_B_to_A,

    # Number of arrivals
    "arrivals_A": num_arrivals_A,
    "arrivals_B": num_arrivals_B,
    "arrivals_C": num_arrivals_C,

    "Total_arrivals": len(arrivals_A) + len(arrivals_B) + len(arrivals_C),

    # Probability ward is full when patient arrives - also includes B-arrivals transferred to ward A
    "P_full_A": ward_full_A / len(arrivals_A) if arrivals_A else 0, # same as all_bed_occ_A
    "P_full_B": ward_full_B / len(arrivals_B) if arrivals_B else 0,
    "P_full_C": ward_full_C / len(arrivals_C) if arrivals_C else 0, # same as all_bed_occ_C

    # Probability a B-arrival patient gets sent to another hospital
    "P_B_to_other_hospital": blocked_B / len(arrivals_B) if len(arrivals_B) > 0 else 0, # same as all_bed_occ_B

    # Relocations - either internally or to another hospital
    "reloc_A": blocked_A,
    "reloc_B": blocked_B + transferred_B_to_A,
    "reloc_C": blocked_C,

    # Total relocated
    "Total_reloc": transferred_B_to_A + blocked_A + blocked_B + blocked_C, # same as reloc_all

    # Utilization
    "util_A": util_A,
    "util_B": util_B,
    "util_C": util_C
}

In [196]:
# Multiple replications
n_replications = 1000

results = []

for _ in tqdm(range(n_replications), desc="Simulating hospital"):

    results.append(simulate_hospital(bedsA, bedsB, bedsC, T))

Simulating hospital: 100%|██████████| 1000/1000 [00:29<00:00, 33.86it/s]


In [197]:
# Results and confidence intervals

metric_names = [
    "arrivals_A",
    "arrivals_B",
    "arrivals_C",
    "Total_arrivals",

    "blocked_A",
    "blocked_B",
    "blocked_C",
    "Total_blocked",

    "transferred_B_to_A",

    "P_full_A",
    "P_full_B",
    "P_full_C",

    "P_B_to_other_hospital",

    "reloc_A",
    "reloc_B",
    "reloc_C",
    "Total_reloc",

    "util_A",
    "util_B",
    "util_C"
]

z = 1.96

for metric in metric_names:

    values = np.array([r[metric] for r in results], dtype=float)

    mean_val = np.mean(values)

    std_val = np.std(values, ddof=1)

    se = std_val / np.sqrt(n_replications)

    ci_low = mean_val - z * se
    ci_high = mean_val + z * se

    print(
        f"{metric}: "
        f"{mean_val:.4f} "
        f"(95% CI: [{ci_low:.4f}, {ci_high:.4f}])"
    )

arrivals_A: 2070.6370 (95% CI: [2043.9500, 2097.3240])
arrivals_B: 415.7430 (95% CI: [410.0985, 421.3875])
arrivals_C: 2191.6800 (95% CI: [2188.6990, 2194.6610])
Total_arrivals: 4678.0600 (95% CI: [4650.1784, 4705.9416])
blocked_A: 1329.5510 (95% CI: [1312.0585, 1347.0435])
blocked_B: 34.9670 (95% CI: [33.9908, 35.9432])
blocked_C: 935.2340 (95% CI: [931.8342, 938.6338])
Total_blocked: 2299.7520 (95% CI: [2281.4978, 2318.0062])
transferred_B_to_A: 13.9700 (95% CI: [13.3981, 14.5419])
P_full_A: 0.6261 (95% CI: [0.6200, 0.6321])
P_full_B: 0.1123 (95% CI: [0.1098, 0.1148])
P_full_C: 0.4265 (95% CI: [0.4253, 0.4276])
P_B_to_other_hospital: 0.0802 (95% CI: [0.0781, 0.0823])
reloc_A: 1329.5510 (95% CI: [1312.0585, 1347.0435])
reloc_B: 48.9370 (95% CI: [47.7728, 50.1012])
reloc_C: 935.2340 (95% CI: [931.8342, 938.6338])
Total_reloc: 2313.7220 (95% CI: [2295.7962, 2331.6478])
util_A: 0.8177 (95% CI: [0.8081, 0.8273])
util_B: 0.5933 (95% CI: [0.5853, 0.6013])
util_C: 0.9480 (95% CI: [0.9477, 0.

### Inverse Transform Method

In [257]:
# Arrival intensity functions

# Ward A
def LambdaA(t):
    # Integrated arrival rate function (non-homogeneous poisson process)
    return -(t**3)/10950 + (t**2)/20


# Inverse of cumulative intensity (maps poisson event count -> time)
def LambdaA_inv(s):
    f = lambda t: LambdaA(t) - s
    return brentq(f, 0, 365)


# Ward B intensity is proportional to Ward A
def LambdaB(t):
    return LambdaA(t) / 5


def LambdaB_inv(s):
    f = lambda t: LambdaB(t) - s
    return brentq(f, 0, 365)


# Ward C has constant arrival rate
def rateC(t):
    return 6

In [258]:
# Simulation function

def simulate_hospital(bedsA, bedsB, bedsC, T):

    # Generate arrivals (non-homogenous poisson)
    arrivals_A = []
    S = 0  # Poisson event counter (on cumulative scale)

    # Generate arrival times for Ward A using inversion method
    while True:
        S += rng.exponential(1)

        if S > LambdaA(T):
            break

        t = LambdaA_inv(S)

        if t > T:
            break

        arrivals_A.append(t)

    # Ward B arrivals (scaled intensity)
    arrivals_B = []
    S = 0

    while True:
        S += rng.exponential(1)

        if S > LambdaB(T):
            break

        t = LambdaB_inv(S)

        if t > T:
            break

        arrivals_B.append(t)

    # Ward C arrivals
    arrivals_C = []
    t = 0.0

    while t < T:
        t += rng.exponential(scale=1.0 / rateC(t))

        if t < T:
            arrivals_C.append(t)

    # Make event list
    events = []

    for t in arrivals_A:
        events.append((t, 'A'))

    for t in arrivals_B:
        events.append((t, 'B'))

    for t in arrivals_C:
        events.append((t, 'C'))

    # Sort all events chronologically
    events.sort(key=lambda x: x[0])

    # System state variables
    deps_A = []
    deps_B = []
    deps_C = []

    # Blocking counters
    blocked_A = 0
    blocked_B = 0
    blocked_C = 0

    # Ward-full counters
    ward_full_A = 0
    ward_full_B = 0
    ward_full_C = 0

    # Overflow counter
    transferred_B_to_A = 0

    # Utilization statistics
    area_A = 0.0
    area_B = 0.0
    area_C = 0.0

    last_t = 0.0

    # Main event loop
    for t, ward in events:

        # Update time-weighted occupancy (integrals)
        dt = t - last_t

        area_A += len(deps_A) * dt
        area_B += len(deps_B) * dt
        area_C += len(deps_C) * dt

        last_t = t

        # Remove completed discharges (before arrival)
        while deps_A and deps_A[0] <= t:
            heapq.heappop(deps_A)

        while deps_B and deps_B[0] <= t:
            heapq.heappop(deps_B)

        while deps_C and deps_C[0] <= t:
            heapq.heappop(deps_C)

        # Ward A arrival
        if ward == 'A':

            # Ward A full
            if len(deps_A) >= bedsA:
                ward_full_A += 1

            if len(deps_A) < bedsA:

                los = rng.lognormal(muA, sigmaA)
                heapq.heappush(deps_A, t + los)

            else:

                blocked_A += 1

        # Ward B arrival (ICU + overflow logic)
        elif ward == 'B':

            # Ward B full
            if len(deps_B) >= bedsB:
                ward_full_B += 1

            if len(deps_B) < bedsB:

                los = rng.lognormal(muB, sigmaB)
                heapq.heappush(deps_B, t + los)

            elif len(deps_A) < bedsA:

                # Overflow ICU patient into Ward A
                los = rng.lognormal(muB, sigmaB)
                heapq.heappush(deps_A, t + los)

                transferred_B_to_A += 1

            else:

                blocked_B += 1

        # Ward C arrival
        elif ward == 'C':

            # Ward C full
            if len(deps_C) >= bedsC:
                ward_full_C += 1

            if len(deps_C) < bedsC:

                los = rng.lognormal(muC, sigmaC)
                heapq.heappush(deps_C, t + los)

            else:

                blocked_C += 1

    
    # Compute utilizations
    util_A = area_A / (bedsA * T)
    util_B = area_B / (bedsB * T)
    util_C = area_C / (bedsC * T)

    
    # Return performance measures
    return {

        # Patients who were blocked entirely - could not be admitted to the hospital
        "blocked_A": blocked_A,
        "blocked_B": blocked_B,
        "blocked_C": blocked_C,

        # Total blocked
        "Total_blocked": blocked_A + blocked_B + blocked_C,

        # Overflow ICU patients - tranfered from B to A
        "transferred_B_to_A": transferred_B_to_A,

        # Arrival counts
        "arrivals_A": len(arrivals_A),
        "arrivals_B": len(arrivals_B),
        "arrivals_C": len(arrivals_C),

        # Total arrivals
        "Total_arrivals": len(arrivals_A) + len(arrivals_B) + len(arrivals_C),

        # Probability ward is full when patient arrives - also includes B-arrivals transferred to ward A
        "P_full_A": ward_full_A / len(arrivals_A) if len(arrivals_A) > 0 else 0, # same as all_bed_occ_A
        "P_full_B": ward_full_B / len(arrivals_B) if len(arrivals_B) > 0 else 0,
        "P_full_C": ward_full_C / len(arrivals_C) if len(arrivals_C) > 0 else 0, # same as all_bed_occ_C

        # Probability a B-arrival patient gets sent to another hospital
        "P_B_to_other_hospital": blocked_B / len(arrivals_B) if len(arrivals_B) > 0 else 0, # same as all_bed_occ_B

        # Relocations - either internally or to another hospital
        "reloc_A": blocked_A, 
        "reloc_B": blocked_B + transferred_B_to_A,
        "reloc_C": blocked_C,

        # Total relocated
        "Total_reloc": transferred_B_to_A + blocked_A + blocked_B + blocked_C, # same as reloc_all
  
        # Utilization fractions
        "util_A": util_A,
        "util_B": util_B,
        "util_C": util_C
    }

In [259]:
# Multiple replications
n_replications = 1000

results = []

for _ in tqdm(range(n_replications), desc="Simulating hospital"):

    results.append(simulate_hospital(bedsA, bedsB, bedsC, T))

Simulating hospital: 100%|██████████| 1000/1000 [00:57<00:00, 17.44it/s]


In [261]:
# Results and confidence intervals

metric_names = [
    "arrivals_A",
    "arrivals_B",
    "arrivals_C",
    "Total_arrivals",

    "blocked_A",
    "blocked_B",
    "blocked_C",
    "Total_blocked",

    "transferred_B_to_A",

    "P_full_A",
    "P_full_B",
    "P_full_C",

    "P_B_to_other_hospital",

    "reloc_A",
    "reloc_B",
    "reloc_C",
    "Total_reloc",

    "util_A",
    "util_B",
    "util_C"
]

z = 1.96

for metric in metric_names:

    values = np.array([r[metric] for r in results], dtype=float)

    mean_val = np.mean(values)

    std_val = np.std(values, ddof=1)

    se = std_val / np.sqrt(n_replications)

    ci_low = mean_val - z * se
    ci_high = mean_val + z * se

    print(
        f"{metric}: "
        f"{mean_val:.4f} "
        f"(95% CI: [{ci_low:.4f}, {ci_high:.4f}])"
    )

arrivals_A: 2221.6790 (95% CI: [2218.7359, 2224.6221])
arrivals_B: 443.5600 (95% CI: [442.2496, 444.8704])
arrivals_C: 2190.5470 (95% CI: [2187.7681, 2193.3259])
Total_arrivals: 4855.7860 (95% CI: [4851.5354, 4860.0366])
blocked_A: 1421.4820 (95% CI: [1418.2240, 1424.7400])
blocked_B: 38.5640 (95% CI: [37.7900, 39.3380])
blocked_C: 936.0080 (95% CI: [932.6624, 939.3536])
Total_blocked: 2396.0540 (95% CI: [2391.1879, 2400.9201])
transferred_B_to_A: 13.1800 (95% CI: [12.8680, 13.4920])
P_full_A: 0.6397 (95% CI: [0.6388, 0.6406])
P_full_B: 0.1158 (95% CI: [0.1138, 0.1178])
P_full_C: 0.4271 (95% CI: [0.4259, 0.4282])
P_B_to_other_hospital: 0.0863 (95% CI: [0.0847, 0.0879])
reloc_A: 1421.4820 (95% CI: [1418.2240, 1424.7400])
reloc_B: 51.7440 (95% CI: [50.7592, 52.7288])
reloc_C: 936.0080 (95% CI: [932.6624, 939.3536])
Total_reloc: 2409.2340 (95% CI: [2404.3095, 2414.1585])
util_A: 0.8999 (95% CI: [0.8993, 0.9005])
util_B: 0.6429 (95% CI: [0.6411, 0.6447])
util_C: 0.9640 (95% CI: [0.9638, 0.

## Optimization

In [245]:
# Grid search to find optimal distribution of beds (only 1 replication)

results = []

for A in range(1, 76):
    for B in range(1, 76):

        C = 75 - A - B
        if C <= 0:
            continue

        metrics = simulate_hospital(A, B, C, T=365)

        # Considers the sum of the relocated patients
        score = (metrics["Total_reloc"])

        results.append({"A": A, "B": B, "C": C, "score": score, **metrics})

# Convert to DataFrame
df = pd.DataFrame(results)

# Best configuration
best = df.loc[df["score"].idxmin()]

print(best)

A                          20.000000
B                          12.000000
C                          43.000000
score                    2046.000000
blocked_A                1388.000000
blocked_B                 112.000000
blocked_C                 511.000000
Total_blocked            2011.000000
transferred_B_to_A         35.000000
arrivals_A               2145.000000
arrivals_B                437.000000
arrivals_C               2055.000000
Total_arrivals           4637.000000
P_full_A                    0.647086
P_full_B                    0.336384
P_full_C                    0.248662
P_B_to_other_hospital       0.256293
reloc_A                  1388.000000
reloc_B                   147.000000
reloc_C                   511.000000
Total_reloc              2046.000000
util_A                      0.911653
util_B                      0.778857
util_C                      0.935597
Name: 1227, dtype: float64


## Sensitivity Analysis

In [ ]:
# Test exponential LOS distribution

# Bed distribution - using the results from grid search
total_beds = 75

bedsA = 20
bedsB = 12
bedsC = 43

# Simulation function
def simulate_hospital(bedsA, bedsB, bedsC, T):

    # Generate arrivals (non-homogenous poisson)
    arrivals_A = []
    S = 0  # Poisson event counter (on cumulative scale)

    # Precompute mean LOS for exponential distribution (keeps same mean as lognormal)
    meanA = np.exp(muA + sigmaA**2 / 2)
    meanB = np.exp(muB + sigmaB**2 / 2)
    meanC = np.exp(muC + sigmaC**2 / 2)

    # Generate arrival times for Ward A using inversion method
    while True:
        S += rng.exponential(1)

        if S > LambdaA(T):
            break

        t = LambdaA_inv(S)

        if t > T:
            break

        arrivals_A.append(t)

    # Ward B arrivals (scaled intensity)
    arrivals_B = []
    S = 0

    while True:
        S += rng.exponential(1)

        if S > LambdaB(T):
            break

        t = LambdaB_inv(S)

        if t > T:
            break

        arrivals_B.append(t)

    # Ward C arrivals
    arrivals_C = []
    t = 0.0

    while t < T:
        t += rng.exponential(scale=1.0 / rateC(t))

        if t < T:
            arrivals_C.append(t)

    # Make event list
    events = []

    for t in arrivals_A:
        events.append((t, 'A'))

    for t in arrivals_B:
        events.append((t, 'B'))

    for t in arrivals_C:
        events.append((t, 'C'))

    # Sort all events chronologically
    events.sort(key=lambda x: x[0])

    # System state variables
    deps_A = []
    deps_B = []
    deps_C = []

    # Blocking counters
    blocked_A = 0
    blocked_B = 0
    blocked_C = 0

    # Ward-full counters
    ward_full_A = 0
    ward_full_B = 0
    ward_full_C = 0

    # Overflow counter
    transferred_B_to_A = 0

    # Utilization statistics
    area_A = 0.0
    area_B = 0.0
    area_C = 0.0

    last_t = 0.0

    # Main event loop
    for t, ward in events:

        # Update time-weighted occupancy (integrals)
        dt = t - last_t

        area_A += len(deps_A) * dt
        area_B += len(deps_B) * dt
        area_C += len(deps_C) * dt

        last_t = t

        # Remove completed discharges (before arrival)
        while deps_A and deps_A[0] <= t:
            heapq.heappop(deps_A)

        while deps_B and deps_B[0] <= t:
            heapq.heappop(deps_B)

        while deps_C and deps_C[0] <= t:
            heapq.heappop(deps_C)

        # Ward A arrival
        if ward == 'A':

            # Ward A full
            if len(deps_A) >= bedsA:
                ward_full_A += 1

            if len(deps_A) < bedsA:

                los = rng.exponential(meanA)
                heapq.heappush(deps_A, t + los)

            else:

                blocked_A += 1

        # Ward B arrival (ICU + overflow logic)
        elif ward == 'B':

            # Ward B full
            if len(deps_B) >= bedsB:
                ward_full_B += 1

            if len(deps_B) < bedsB:

                los = rng.exponential(meanB)
                heapq.heappush(deps_B, t + los)

            elif len(deps_A) < bedsA:

                # Overflow ICU patient into Ward A
                los = rng.exponential(meanB)
                heapq.heappush(deps_A, t + los)

                transferred_B_to_A += 1

            else:

                blocked_B += 1

        # Ward C arrival
        elif ward == 'C':

            # Ward C full
            if len(deps_C) >= bedsC:
                ward_full_C += 1

            if len(deps_C) < bedsC:

                los = rng.exponential(meanC)
                heapq.heappush(deps_C, t + los)

            else:

                blocked_C += 1

    
    # Account for time from last event until end of simulation
    dt = T - last_t

    area_A += len(deps_A) * dt
    area_B += len(deps_B) * dt
    area_C += len(deps_C) * dt
    
    # Compute utilizations
    util_A = area_A / (bedsA * T)
    util_B = area_B / (bedsB * T)
    util_C = area_C / (bedsC * T)

    
    # Return performance measures
    return {

        # Patients who were blocked entirely - could not be admitted to the hospital
        "blocked_A": blocked_A,
        "blocked_B": blocked_B,
        "blocked_C": blocked_C,

        # Total blocked
        "Total_blocked": blocked_A + blocked_B + blocked_C,

        # Overflow ICU patients - tranfered from B to A
        "transferred_B_to_A": transferred_B_to_A,

        # Arrival counts
        "arrivals_A": len(arrivals_A),
        "arrivals_B": len(arrivals_B),
        "arrivals_C": len(arrivals_C),

        # Total arrivals
        "Total_arrivals": len(arrivals_A) + len(arrivals_B) + len(arrivals_C),

        # Probability ward is full when patient arrives - also includes B-arrivals transferred to ward A
        "P_full_A": ward_full_A / len(arrivals_A) if len(arrivals_A) > 0 else 0, # same as all_bed_occ_A
        "P_full_B": ward_full_B / len(arrivals_B) if len(arrivals_B) > 0 else 0,
        "P_full_C": ward_full_C / len(arrivals_C) if len(arrivals_C) > 0 else 0, # same as all_bed_occ_C

        # Probability a B-arrival patient gets sent to another hospital
        "P_B_to_other_hospital": blocked_B / len(arrivals_B) if len(arrivals_B) > 0 else 0, # same as all_bed_occ_B

        # Relocations - either internally or to another hospital
        "reloc_A": blocked_A, 
        "reloc_B": blocked_B + transferred_B_to_A,
        "reloc_C": blocked_C,

        # Total relocated
        "Total_reloc": transferred_B_to_A + blocked_A + blocked_B + blocked_C, # same as reloc_all
  
        # Utilization fractions
        "util_A": util_A,
        "util_B": util_B,
        "util_C": util_C
    }

In [247]:
# Multiple replications
n_replications = 1000

results = []

for _ in tqdm(range(n_replications), desc="Simulating hospital"):

    results.append(simulate_hospital(bedsA, bedsB, bedsC, T, ))

Simulating hospital: 100%|██████████| 1000/1000 [00:55<00:00, 18.09it/s]


In [248]:
# Results and confidence intervals

metric_names = [
    "arrivals_A",
    "arrivals_B",
    "arrivals_C",
    "Total_arrivals",

    "blocked_A",
    "blocked_B",
    "blocked_C",
    "Total_blocked",

    "transferred_B_to_A",

    "P_full_A",
    "P_full_B",
    "P_full_C",

    "P_B_to_other_hospital",

    "reloc_A",
    "reloc_B",
    "reloc_C",
    "Total_reloc",

    "util_A",
    "util_B",
    "util_C"
]

z = 1.96

for metric in metric_names:

    values = np.array([r[metric] for r in results], dtype=float)

    mean_val = np.mean(values)

    std_val = np.std(values, ddof=1)

    se = std_val / np.sqrt(n_replications)

    ci_low = mean_val - z * se
    ci_high = mean_val + z * se

    print(
        f"{metric}: "
        f"{mean_val:.4f} "
        f"(95% CI: [{ci_low:.4f}, {ci_high:.4f}])"
    )

arrivals_A: 2222.0660 (95% CI: [2219.1950, 2224.9370])
arrivals_B: 444.8870 (95% CI: [443.5791, 446.1949])
arrivals_C: 2190.5870 (95% CI: [2187.5105, 2193.6635])
Total_arrivals: 4857.5400 (95% CI: [4853.1196, 4861.9604])
blocked_A: 1466.9730 (95% CI: [1463.7396, 1470.2064])
blocked_B: 122.9960 (95% CI: [121.9259, 124.0661])
blocked_C: 669.6960 (95% CI: [666.1151, 673.2769])
Total_blocked: 2259.6650 (95% CI: [2254.4978, 2264.8322])
transferred_B_to_A: 42.7320 (95% CI: [42.2442, 43.2198])
P_full_A: 0.6601 (95% CI: [0.6592, 0.6609])
P_full_B: 0.3716 (95% CI: [0.3693, 0.3738])
P_full_C: 0.3054 (95% CI: [0.3041, 0.3067])
P_B_to_other_hospital: 0.2757 (95% CI: [0.2738, 0.2776])
reloc_A: 1466.9730 (95% CI: [1463.7396, 1470.2064])
reloc_B: 165.7280 (95% CI: [164.3924, 167.0636])
reloc_C: 669.6960 (95% CI: [666.1151, 673.2769])
Total_reloc: 2302.3970 (95% CI: [2297.1801, 2307.6139])
util_A: 0.9018 (95% CI: [0.9011, 0.9024])
util_B: 0.7626 (95% CI: [0.7611, 0.7641])
util_C: 0.9499 (95% CI: [0.94

In [250]:
# Testing different amounts of beds in the system

# Baseline bed distribution
total_base = 75

# Numbers found from grid search
base_A = 20
base_B = 12
base_C = 43

pA = base_A / total_base
pB = base_B / total_base
pC = base_C / total_base


# Function to scale beds
def allocate_beds(total_beds):
    A = int(round(pA * total_beds))
    B = int(round(pB * total_beds))
    C = total_beds - A - B
    return A, B, C


# Bed scenarios
bed_scenarios = [60, 75, 80, 100]

results = []


# Simulation loop
for total in bed_scenarios:

    bedsA, bedsB, bedsC = allocate_beds(total)

    metrics = simulate_hospital(bedsA, bedsB, bedsC, T=365)

    results.append({
        "Total_beds": total,
        "bedsA": bedsA,
        "bedsB": bedsB,
        "bedsC": bedsC,

        # Arrival counts
        "arrivals_A": metrics["arrivals_A"],
        "arrivals_B": metrics["arrivals_B"],
        "arrivals_C": metrics["arrivals_C"],
        "Total_arrivals": metrics["Total_arrivals"],

        # Blocking counts
        "blocked_A": metrics["blocked_A"],
        "blocked_B": metrics["blocked_B"],
        "blocked_C": metrics["blocked_C"],
        "Total_blocked": metrics["Total_blocked"],

        # Overflow ICU patients
        "transferred_B_to_A": metrics["transferred_B_to_A"],

        # Probability ward is full when patient arrives
        "P_full_A": metrics["P_full_A"],
        "P_full_B": metrics["P_full_B"],
        "P_full_C": metrics["P_full_C"],

        # Probability a B-arrival patient gets sent to another hospital
        "P_B_to_other_hospital": metrics["P_B_to_other_hospital"],

        # Relocations - either internally or to another hospital
        "reloc_A": metrics["reloc_A"],
        "reloc_B": metrics["reloc_B"],
        "reloc_C": metrics["reloc_C"],
        "Total_reloc": metrics["Total_reloc"],

        # Utilization fractions
        "util_A": metrics["util_A"],
        "util_B": metrics["util_B"],
        "util_C": metrics["util_C"]
    })


# Create results table
df = pd.DataFrame(results)

print(df)


# Results
df_sorted = df.sort_values("Total_blocked")

print("\n--- Sorted by Total Blocked ---\n")
print(df_sorted.to_string())

   Total_beds  bedsA  bedsB  bedsC  arrivals_A  arrivals_B  arrivals_C  \
0          60     16     10     34        2242         459        2176   
1          75     20     12     43        2268         452        2173   
2          80     21     13     46        2246         469        2230   
3         100     27     16     57        2256         439        2178   

   Total_arrivals  blocked_A  blocked_B  ...  P_full_B  P_full_C  \
0            4877       1612        155  ...  0.409586  0.452665   
1            4893       1515        122  ...  0.373894  0.329038   
2            4945       1425        119  ...  0.356077  0.286547   
3            4873       1253         51  ...  0.184510  0.129477   

   P_B_to_other_hospital  reloc_A  reloc_B  reloc_C  Total_reloc    util_A  \
0               0.337691     1612      188      985         2785  0.927103   
1               0.269912     1515      169      715         2399  0.900213   
2               0.253731     1425      167      639   